In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%run ./_local_config

In [0]:
import pandas as pd
import json
from azure.storage.blob import BlobServiceClient

conn_str = f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};AccountKey={storage_account_key};EndpointSuffix=core.windows.net"

blob_service = BlobServiceClient.from_connection_string(conn_str)
blob_client = blob_service.get_blob_client(container="bronze", blob="erp/battery/battery.json")

stream = blob_client.download_blob().readall()

# Decode bytes to text, split into lines, parse each line as its own JSON object
# (each line here is one full API page response, not one sales record)
text = stream.decode("utf-8-sig")
pages = [json.loads(line) for line in text.splitlines() if line.strip()]

# Each page has a "value" key containing a list of actual sales records —
# flatten all pages into a single list of records
all_records = []
for page in pages:
    all_records.extend(page["value"])

bronze = pd.DataFrame(all_records)

print(bronze.shape)
print(bronze.columns.tolist())
bronze.head()

In [0]:
print(bronze.shape)  # expect something in the hundreds of thousands, matching your earlier ~596K count
print(bronze["itemCategoryCode"].unique())

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.transform.clean_silver import clean_to_silver

silver = clean_to_silver(bronze)
print(silver.shape)
silver.head()

In [0]:
unique_categories = sorted(silver["itemCategoryCode"].dropna().unique())
print(f"Found {len(unique_categories)} unique item categories")
print(unique_categories)

In [0]:
vehicle_counts = silver["vehicle_type"].value_counts(dropna=False)
print(vehicle_counts)

In [0]:
unique_vehicle_types = sorted(silver["vehicle_type"].dropna().unique())

vehicle_type_lookup = pd.DataFrame({
    "vehicle_type_id": [f"V{i}" for i in range(1, len(unique_vehicle_types) + 1)],
    "vehicle_type": unique_vehicle_types
})

print(vehicle_type_lookup)

In [0]:
brand_counts = silver.groupby(["brand_code", "brand_description"]).size().reset_index(name="count")
brand_counts = brand_counts.sort_values("count", ascending=False)
print(brand_counts.to_string())

In [0]:
import json

# Convert DataFrame to JSON records, handling datetime serialization
silver_json = silver.to_json(orient="records", date_format="iso", lines=False)

blob_client = blob_service.get_blob_client(
    container="silver",
    blob="erp/battery/battery_clean.json"
)
blob_client.upload_blob(silver_json, overwrite=True)

print(f"Saved {len(silver)} rows to silver/erp/battery/battery_clean.json")

In [0]:
container_client = blob_service.get_container_client("silver")
for blob in container_client.list_blobs(name_starts_with="erp/battery/"):
    print(blob.name, blob.size)

In [ ]:
missing_dates = set(pd.date_range(silver["posting_date"].min(), silver["posting_date"].max())) - set(silver["posting_date"].unique())
missing_df = pd.DataFrame({"missing_date": sorted(missing_dates)})
missing_df["day_of_week"] = pd.to_datetime(missing_df["missing_date"]).dt.day_name()

print(f"Total missing days: {len(missing_df)}")
print(missing_df["day_of_week"].value_counts())